In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
# from model.systemModel                  import SystemModel as systemModel
from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-11-29 12:19:50.639217: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-29 12:19:50.778096: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-29 12:19:51.557883: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/ducan/anaconda3/envs/tensorflow2.10.0Crusher/lib/
2024-11-29 12:19:51.557971: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerro

##### HI COMPONENT AND SYSTEM

In [2]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data

cr1, cr2, cv2           = preData.loadH5File(name = "WITHOUTNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file



# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(cr1)
withHourCr2             = preData.addHours(cr2)
withHourCv2             = preData.addHours(cv2)


# --------------------------------
# Use for system HI
# --------------------------------
insertedCr1Sys, insertedCr2Sys,  insertedCv2Sys     = preData.insertForTheSameLength(data         = [withHourCr1.copy(),withHourCr2.copy(),withHourCv2.copy()], 
                                                                                    validColumn  = [preData.crusher1Value, preData.crusher2Value, preData.conveyorValue + preData.crusher2ConveyorValue])

withIDCr1Sys, withIDCr2Sys, withIDCv2Sys, IdSysSys  = preData.attachIDPerEvent(df = [insertedCr1Sys,insertedCr2Sys,insertedCv2Sys], mainternanceAsCriticalSd = True)

insertedCr1 = withHourCr1
insertedCr2 = withHourCr2
insertedCv2 = withHourCv2

withIDCr1, withIDCr2, withIDCv2, IdSys = preData.attachIDPerEvent(df = [insertedCr1,insertedCr2,insertedCv2], mainternanceAsCriticalSd = True)


addedData               = preData.addition_var(withIDCr1) 


handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData, headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1)                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data

# -----------------------------
# Add the cycle and RUL of system
systemData                  = pd.DataFrame()
systemData['hours']         = withTimeDataCr1["hours"]
systemData['trajectoryID']  = IdSys
withTimeDataSys             = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
dataCr1                 = withTimeDataCr1[handledValues]
cycleCr1                = withTimeDataCr1["cycle"]
rulCr1                  = withTimeDataCr1["rul"]
timeCr1                 = withTimeDataCr1["hours"]
trajectIDCr1            = withTimeDataCr1["trajectoryID"]

dataCr2                 = withTimeDataCr2[preData.crusher2Value]
cycleCr2                = withTimeDataCr2["cycle"]
rulCr2                  = withTimeDataCr2["rul"]
timeCr2                 = withTimeDataCr2["hours"]
trajectIDCr2            = withTimeDataCr2["trajectoryID"]

dataCv2                 = withTimeDataCv2[preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["cycle"]
rulCv2                  = withTimeDataCv2["rul"]
timeCv2                 = withTimeDataCv2["hours"]
trajectIDCv2            = withTimeDataCv2["trajectoryID"]

systemRUL               = withTimeDataSys["rul"]


print(timeCr1)
print(timeCr2)
print(timeCv2)
print(insertedCr1Sys["hours"])



Process:28112||2024-11-29 12:19:53,274-INFO [preProcessingData.py:loadH5File():575]:Loading H5 file ...
Process:28112||2024-11-29 12:19:54,550-INFO [preProcessingData.py:loadH5File():581]:Finish loading data from files ...
Process:28112||2024-11-29 12:19:58,443-INFO [preProcessingData.py:attachIDPerEvent():422]:Attaching ID into trajectories
Process:28112||2024-11-29 12:20:17,803-INFO [preProcessingData.py:attachIDPerEvent():422]:Attaching ID into trajectories
Process:28112||2024-11-29 12:20:35,099-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....
Process:28112||2024-11-29 12:20:35,660-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....
Process:28112||2024-11-29 12:20:36,107-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....


0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
321370    8759.933
321371    8759.950
321372    8759.967
321373    8759.983
321374    8760.000
Name: hours, Length: 321375, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
466645    8759.933
466646    8759.950
466647    8759.967
466648    8759.983
466649    8760.000
Name: hours, Length: 466650, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
451063    8759.933
451064    8759.950
451065    8759.967
451066    8759.983
451067    8760.000
Name: hours, Length: 451068, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dty

###### Crusher1

In [3]:
myConfig["inFeatureDim"] = 41

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr1"]))  # Batch size is None, feature size is 32()
model.summary()
model.load_weights(model_dir = "checkpoint/crusher1BestTAE/")

2024-11-29 12:20:37.027859: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:20:37.028068: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:20:37.034355: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:20:37.034582: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:20:37.034766: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 0 (unused)
 ection)                                                         
                                                                 
 encoder (Functional)        [(None, 1),               122306    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 41)            121941    
                                                                 
 hi_system_and_rul (HiSystem  multiple                 6413      
 AndRul)                                                         
                                                                 
Total params: 250,712
Trainable params: 250,70

Process:28112||2024-11-29 12:20:38,884-INFO [componentModel.py:load_weights():780]:Loading the model at the direction: checkpoint/crusher1BestTAE/
Process:28112||2024-11-29 12:20:38,915-INFO [componentModel.py:load_weights():790]:Loaded the model at the direction:checkpoint/crusher1BestTAE/ successfully!!!


In [4]:
results             = {}
time                = {}
results["cr1"]      = []
results["rulCr1"]   = []
time["timeCr1"]     = []


probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCr1["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1, seqTimeCr1                  = preData.getTensorData(dataList   =    [dataCr1, timeCr1], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1)
    # print(predictedRul)
    results["cr1"].append(pd.Series(tf.squeeze(zCr1).numpy()))
    results["rulCr1"].append(predictedRul)
    time["timeCr1"].append(seqTimeCr1[:,-1,:])

print(time["timeCr1"])




Process:28112||2024-11-29 12:20:38,927-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


0


2024-11-29 12:20:39.354360: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 104393216 exceeds 10% of free system memory.
2024-11-29 12:20:49.473149: W tensorflow/core/common_runtime/bfc_allocator.cc:479] Allocator (GPU_0_bfc) ran out of memory trying to allocate 151.76MiB (rounded to 159136000)requested by op Conv2D
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2024-11-29 12:20:49.473209: I tensorflow/core/common_runtime/bfc_allocator.cc:1033] BFCAllocator dump for GPU_0_bfc
2024-11-29 12:20:49.473232: I tensorflow/core/common_runtime/bfc_allocator.cc:1040] Bin (256): 	Total Chunks: 46, Chunks in use: 44. 11.5KiB allocated for chunks. 11.0KiB in use in bin. 1004B client-requested in use in bin.
2024-11-29 12:20:49.473249: I tensorflow/core/common_runtime/bfc_allocator.cc:1040] Bin (512): 	Total Chunks: 6, Chu

ResourceExhaustedError: Exception encountered when calling layer "enc_conv_0" "                 f"(type Conv1D).

{{function_node __wrapped__Conv2D_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating tensor with shape[4973,1,32,250] and type float on /job:localhost/replica:0/task:0/device:GPU:0 by allocator GPU_0_bfc [Op:Conv2D]

Call arguments received by layer "enc_conv_0" "                 f"(type Conv1D):
  • inputs=tf.Tensor(shape=(4973, 64, 41), dtype=float32)

In [ ]:
print(time["timeCr1"][0].shape)


(4973, 1)


###### Crusher2

In [ ]:
myConfig["inFeatureDim"] = 30

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr2"]))  # Batch size is None, feature size is 32()
model.summary()
model.load_weights(model_dir = "checkpoint/crusher2BestTAE/")

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection_1 (SensorS  multiple                 0 (unused)
 election)                                                       
                                                                 
 encoder (Functional)        [(None, 1),               114056    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 30)            113680    
                                                                 
 hi_system_and_rul_1 (HiSyst  multiple                 6413      
 emAndRul)                                                       
                                                                 
Total params: 234,190
Trainable params: 234,

Process:16999||2024-11-29 09:28:52,550-INFO [componentModel.py:load_weights():780]:Loading the model at the direction: checkpoint/bestModel38/
Process:16999||2024-11-29 09:28:52,571-INFO [componentModel.py:load_weights():790]:Loaded the model at the direction:checkpoint/bestModel38/ successfully!!!


In [ ]:
results["cr2"]      = []
results["rulCr2"]   = []
time["timeCr2"]     = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCr2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr2, seqTimeCr2                    = preData.getTensorData(dataList     =    [dataCr2, timeCr2], 
                                                                    trajectId   = trajectID,
                                                                    seqLength   = preData.config['seqLength'], 
                                                                    overlap     = preData.config['seqLength']-1,
                                                                    getAtIdx    = idx + 1,
                                                                    typeInput   = "listOfPandas")
    
    zCr2, binaryMaskCr2, probsMaskCr2, predictedRul     = model.getHIandRUL(seqDataCr2)
    # print(predictedRul)
    results["cr2"].append(pd.Series(tf.squeeze(zCr2).numpy()))
    results["rulCr2"].append(predictedRul)
    time["timeCr2"].append(seqTimeCr2[:,-1,:])


Process:16999||2024-11-29 09:28:52,582-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


0


Process:16999||2024-11-29 09:28:53,075-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


1


Process:16999||2024-11-29 09:28:54,098-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


2


... (94 outputs left out)

Process:16999||2024-11-29 09:29:23,843-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


57


Process:16999||2024-11-29 09:29:24,526-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


58


###### Conveyorbelt 2

In [ ]:
myConfig["inFeatureDim"] = 31

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCv2"]))  # Batch size is None, feature size is 32()
model.summary()
model.load_weights(model_dir = "checkpoint/CV2BestTAE/")

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection_2 (SensorS  multiple                 0 (unused)
 election)                                                       
                                                                 
 encoder (Functional)        [(None, 1),               114806    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 31)            114431    
                                                                 
 hi_system_and_rul_2 (HiSyst  multiple                 6413      
 emAndRul)                                                       
                                                                 
Total params: 235,692
Trainable params: 235,

Process:16999||2024-11-29 09:29:26,193-INFO [componentModel.py:load_weights():780]:Loading the model at the direction: checkpoint/bestModel41/
Process:16999||2024-11-29 09:29:26,211-INFO [componentModel.py:load_weights():790]:Loaded the model at the direction:checkpoint/bestModel41/ successfully!!!


In [ ]:
results["cv2"]          = []
results["rulCv2"]       = []
time["timeCv2"]         = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCv2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCv2, seqTimeCv2                  = preData.getTensorData(dataList   =    [dataCv2, timeCv2], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCv2, binaryMaskCv2, probsMaskCv2, predictedRul     = model.getHIandRUL(seqDataCv2)
    # print(predictedRul)
    results["cv2"].append(pd.Series(tf.squeeze(zCv2).numpy()))
    results["rulCv2"].append(predictedRul)
    time["timeCv2"].append(seqTimeCv2[:,-1,:])


Process:16999||2024-11-29 09:29:26,224-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


0


Process:16999||2024-11-29 09:29:26,667-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


1


Process:16999||2024-11-29 09:29:27,856-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


2


... (106 outputs left out)

Process:16999||2024-11-29 09:30:03,339-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


79


Process:16999||2024-11-29 09:30:04,067-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


80


###### System

In [ ]:
# # ------------------------------
# # Normalize
# # ------------------------------
# combined_data = pd.concat(results["cr1"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCr1       = MinMaxScaler()
# scalerCr1.fit(data_reshaped)
# results["cr1"]  = [pd.DataFrame(scalerCr1.transform(df.values.reshape(-1, 1))) for df in results["cr1"]]


# combined_data = pd.concat(results["cr2"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCr2       = MinMaxScaler()
# scalerCr2.fit(data_reshaped)
# results["cr2"]  = [pd.DataFrame(scalerCr2.transform(df.values.reshape(-1, 1))) for df in results["cr2"]]

# combined_data = pd.concat(results["cv2"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCv2       = MinMaxScaler()
# scalerCv2.fit(data_reshaped)
# results["cv2"]  = [pd.DataFrame(scalerCv2.transform(df.values.reshape(-1, 1))) for df in results["cv2"]]

In [ ]:
# scalerCr1       = MinMaxScaler()
# norHI1          = scalerCr1.fit_transform(aligned_HIs[1])

# scalerCr2       = MinMaxScaler()
# norHI2          = scalerCr1.fit_transform(aligned_HIs[2])

# scalerCv2       = MinMaxScaler()
# norHI3          = scalerCv2.fit_transform(aligned_HIs[3])

In [ ]:
# ------------------------------
# HI system
all_times = insertedCr1Sys["hours"]

# Create a dictionary to store aligned HIs for each component
aligned_HIs = {1: [], 2: [], 3: []}


def align_hi(HI, timeHI, all_times):
    aligned = pd.Series(index=all_times, dtype=float)       # Initialize with NaN
    for hi, times in zip(HI, timeHI):
        aligned.loc[times[:,0]] = hi.values                 # Align based on the original time
    return aligned

# Align each component's HI

aligned_HIs[1] = align_hi(results["cr1"], time["timeCr1"], all_times)
aligned_HIs[2] = align_hi(results["cr2"], time["timeCr2"], all_times)
aligned_HIs[3] = align_hi(results["cv2"], time["timeCv2"], all_times)




# Combine all aligned HIs into a DataFrame
aligned_df = pd.DataFrame({'Time': all_times,
                           'OriginalHICr1': aligned_HIs[1].values,
                           'OriginalHICr2': aligned_HIs[2].values,
                           'OriginalHICv2': aligned_HIs[3].values})


window_size = 65
df_moving_avg = aligned_df.iloc[:, 1:].rolling(window=window_size, min_periods=1).mean()                    # Moving average
df_normalized = df_moving_avg.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)                  # Normalize
df_normalized = df_normalized.rename(columns={'OriginalHICr1': 'norHICr1', 'OriginalHICr2': 'norHICr2', 'OriginalHICv2': 'norHICv2'})

# Concatenating the Time column back with the moving average
df_result = pd.concat([aligned_df["Time"], df_normalized], axis=1)
# Calculate the system HI (row-wise min ignoring NaN)
df_result['System_HI'] = df_result[['norHICr1', 'norHICr2', 'norHICv2']].min(axis=1, skipna=True)


df_result["OriginalHICr1"] = aligned_df["OriginalHICr1"]
df_result["OriginalHICr2"] = aligned_df["OriginalHICr2"]
df_result["OriginalHICv2"] = aligned_df["OriginalHICv2"]
df_result["trajectoryCr1"] = withIDCr1Sys["trajectoryID"]
df_result["trajectoryCr2"] = withIDCr2Sys["trajectoryID"]
df_result["trajectoryCv2"] = withIDCv2Sys["trajectoryID"]
df_result["trajectorySys"] = IdSysSys

In [ ]:
# df_result.to_excel("HIDataWithTrajectory.xlsx", index=False)

In [ ]:
# Plotting the data with Plotly
import plotly.express as px

fig = px.line(df_result, x="Time", y=["System_HI"], 

              labels={"value": "Health Index", "Time": "Time (s)"}, 

              title="Health Index Over Time")

fig.update_layout(
    plot_bgcolor='white',  # Background of the plot area
    paper_bgcolor='white' # Background of the overall canvas
)

fig.show()

In [ ]:
# Plotting the data with Plotly
import plotly.express as px

fig = px.line(df_result, x="Time", y=["OriginalHICr1", "OriginalHICr2", "OriginalHICv2"], 

              labels={"value": "Health Index", "Time": "Time (s)"}, 

              title="Health Index Over Time")



fig.show()

In [ ]:
fig = {}
for idxResults in aligned_df.columns[1:]:
    fig[idxResults]     = go.Figure()
    current_time        = 0

    for idx, signal in enumerate(results[idxResults]):
        signal          = tf.squeeze(signal).numpy()
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[idxResults].add_trace(go.Scatter(
                                                x   = time_vector,
                                                y   = signal,
                                                mode= 'lines',
                                                name= f'Signal {idx + 1}'  # Label each signal for legend toggling
        ))
    # Customize layout for readability
    fig[idxResults].update_layout(
                                    title       = idxResults,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[idxResults].show()

KeyError: 'OriginalHICr1'

In [ ]:
# Function to plot HI grouped by ID
def plot_hi_by_id(time, hi, ids, title):
    fig = go.Figure()
    df_hi = pd.DataFrame({'Time': time, 'HI': hi, 'ID': ids})
    grouped = df_hi.groupby('ID')
    
    for id_val, group in grouped:
        if not pd.isna(id_val):  # Skip NaN IDs
            fig.add_trace(go.Scatter(
                x=group['Time'],
                y=group['HI'],
                mode='lines+markers',
                name=f'Trajectory {int(id_val)}'
            ))
    
    fig.update_layout(title=title, xaxis_title='Time', yaxis_title='Health Indicator (HI)')
    return fig

# Plot HI1, HI2, HI3
withIDCr1Sys, withIDCr2Sys, withIDCv2Sys, IdSysSys
ID1 = withIDCr1Sys["trajectoryID"]
ID2 = withIDCr2Sys["trajectoryID"]
ID3 = withIDCv2Sys["trajectoryID"]
IDSys = IdSysSys


fig_hi1 = plot_hi_by_id(aligned_df['Time'], aligned_df['HI1'], ID1, "Crusher 1 with Trajectories")
fig_hi2 = plot_hi_by_id(aligned_df['Time'], aligned_df['HI2'], ID2, "Crusehr 2 with Trajectories")
fig_hi3 = plot_hi_by_id(aligned_df['Time'], aligned_df['HI3'], ID3, "Conveyor betl with Trajectories")
fig_hi4 = plot_hi_by_id(aligned_df['Time'], aligned_df['System_HI'], IDSys, "System_HI with Trajectories")

# Plot System HI (no ID grouping)
fig_system_hi = go.Figure()
fig_system_hi.add_trace(go.Scatter(
    x=aligned_df['Time'],
    y=aligned_df['System_HI'],
    mode='lines+markers',
    name='System HI'
))
fig_system_hi.update_layout(title="System HI", xaxis_title='Time', yaxis_title='System Health Indicator (HI)')

# Display all plots
fig_hi1.show()
fig_hi2.show()
fig_hi3.show()
fig_system_hi.show()

In [ ]:
print("OK")

In [ ]:
model       = systemModel(myConfig) 
model.build(input_shape=[(None, myConfig["seqLength"], myConfig["inCr1"]), (None, myConfig["seqLength"], myConfig["inCr2"]), (None, myConfig["seqLength"], myConfig["inCv2"])])  # Batch size is None, feature size is 32()
# model.hiSystemAndRul.build(input_shape=[(None,4,5),(None,4,5),(None,4,5),(None,1,),(None,1,),(None,1,)])
# model.build(input_shape=[(None, myConfig["seqLength"], myConfig["inCr1"]), (None, myConfig["seqLength"], myConfig["inCr2"]), (None, myConfig["seqLength"], myConfig["inCv2"])])  # Batch size is None, feature size is 32()
model.summary()
model.load_weights(model_dir = "checkpoint/bestModel42/")


# (522340, 60, 32) (522340, 1) (522340, 60, 30) (522340, 1) (522340, 60, 2) (522340, 1) (522340, 1)

In [ ]:
results            = {}
results["cr1"]     = []
results["cr2"]     = []
results["cv2"]     = []
results["sys"]     = []
results["rul"]     = []

probsAll        = []
binaryMaskAll   = []
trajectID = IdSys.astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1,seqDataCr2,seqDataCv2         = preData.getTensorData(dataList   =    [dataCr1, dataCr2, dataCv2], 
                                                                    trajectId   = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, zCr2,  zCv2,                              \
    binaryMaskCr1, binaryMaskCr2, binaryMaskCv2,    \
    probsMaskCr1, probsMaskCr2, probsMaskCv2,       \
    hiSystem, predictedRul                          = model.getHIandRUL([seqDataCr1,seqDataCr2,seqDataCv2])
    results["cr1"].append(zCr1)
    results["cr2"].append(zCr2)
    results["cv2"].append(zCv2)
    results["sys"].append(hiSystem)
    results["rul"].append(predictedRul)

### MASK

##### Crusher1

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(preData.crusher1Value):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)


##### Crusher 2

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr2)}")
for idx,idxSS in enumerate(preData.crusher2Value):
    ssTable.add_row([idx,idxSS,probsMaskCr2[idx].numpy(),binaryMaskCr2[idx].numpy()])

print(ssTable)

##### ConveyorBelt 2

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCv2)}")
for idx,idxSS in enumerate(preData.conveyorValue + preData.crusher2ConveyorValue):
    ssTable.add_row([idx,idxSS,probsMaskCv2[idx].numpy(),binaryMaskCv2[idx].numpy()])

print(ssTable)

### Health indicators

##### Crusher 1

In [ ]:
fig = {}
for idxResults in results.keys():
    fig[idxResults]     = go.Figure()
    current_time        = 0

    for idx, signal in enumerate(results[idxResults]):
        signal          = tf.squeeze(signal).numpy()
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[idxResults].add_trace(go.Scatter(
                                                x   = time_vector,
                                                y   = signal,
                                                mode= 'lines',
                                                name= f'Signal {idx + 1}'  # Label each signal for legend toggling
        ))
    # Customize layout for readability
    fig[idxResults].update_layout(
                                    title       = idxResults,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[idxResults].show()


In [ ]:
from model.systemModel                  import SystemModel
from model.systemModel                  import SubclassModelCheckpoint



if preData.config["component"]  == "system":
    insertedCr1, insertedCr2,  insertedCv2   = preData.insertForTheSameLength(data         = [withHourCr1,withHourCr2,withHourCv2], 
                                                                            validColumn  = [preData.crusher1Value, preData.crusher2Value, preData.conveyorValue + preData.crusher2ConveyorValue])
else:
    insertedCr1 = withHourCr1
    insertedCr2 = withHourCr2
    insertedCv2 = withHourCv2



withIDCr1, withIDCr2, withIDCv2, IdSys = preData.attachIDPerEvent(df = [insertedCr1,insertedCr2,insertedCv2], mainternanceAsCriticalSd = True)


addedData               = preData.addition_var(withIDCr1) 


handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData, headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1)                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data

# -----------------------------
# Add the cycle and RUL of system
systemData                  = pd.DataFrame()
systemData['hours']         = withTimeDataCr1["hours"]
systemData['trajectoryID']  = IdSys
withTimeDataSys             = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
dataCr1                 = withTimeDataCr1[handledValues]
cycleCr1                = withTimeDataCr1["cycle"]
rulCr1                  = withTimeDataCr1["rul"]
trajectIDCr1            = withTimeDataCr1["trajectoryID"]

dataCr2                 = withTimeDataCr2[preData.crusher2Value]
cycleCr2                = withTimeDataCr2["cycle"]
rulCr2                  = withTimeDataCr2["rul"]
trajectIDCr2            = withTimeDataCr2["trajectoryID"]

dataCv2                 = withTimeDataCv2[preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["cycle"]
rulCv2                  = withTimeDataCv2["rul"]
trajectIDCv2            = withTimeDataCv2["trajectoryID"]

systemRUL               = withTimeDataSys["rul"]
